# âš¡ S6E9: Smart Titan Meta-Blend (0.94650+)

### ðŸ§  Strategic Synthesis of the Community SOTA
The current public leaderboard ceiling sits at **0.94650**. As proven by multiple independent analyses, the public leaderboard is governed by a concave response surface where arbitrary averaging of weaker models degrades accuracy, while pure anchors retain the peak.

In this notebook, we perform a mathematically guided **Smart Consensus Blend**:
1. **Core Anchors (92%)**: We assemble the consensus of the verified 0.94650 leaders:
   - [@taeyangg4](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) (Multi-Paradigm Lexsort Master)
   - [@talhatursun](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble) (Daily Rank-Average Ensemble)
   - [@chinzorigtganbat](https://www.kaggle.com/code/chinzorigtganbat/s6e9-does-breaking-ties-help) (Tie-Breaking & XGB Exploration)
   - [@nathaliach](https://www.kaggle.com/code/nathaliach/ev-blend) (EV Blend)
2. **Orthogonal Variance Injection (8%)**:
   - [@denpugovkin](https://www.kaggle.com/code/denpugovkin/s6e9-nested-te-lightgbm-original-table-dgp-rej) (Nested-TE LightGBM with DGP Rejection)
   - [@sometimessubodh](https://www.kaggle.com/code/sometimessubodh/s6e9-6-model-meta-stacking-engine) (6-Model Meta-Stacking Engine)
3. **Deterministic Boundary Shift**: Exact alignment with the 4 empirical Data-Generating-Process rules (forcing 3,731 deterministic edge cases to the extreme bounds).
4. **Lexicographical Tie-Breaking**: Ensuring strictly 0 tied rankings across all 286,571 test rows.

*If this notebook is helpful to your work, please consider upvoting the original authors and this notebook!*


In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
warnings.filterwarnings('ignore')

print('1. Locating Datasets & Kernel Sources...')
def find_file(keyword, root='/kaggle/input'):
    hits = glob.glob(os.path.join(root, '**', keyword), recursive=True)
    if not hits:
        # Fallback search
        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                if f.endswith('.csv') and keyword.lower() in f.lower():
                    return os.path.join(dirpath, f)
        raise FileNotFoundError(f'Could not find file matching {keyword} under {root}')
    return hits[0]

test_p = find_file('test.csv')
sample_sub_p = find_file('sample_submission.csv')

test = pd.read_csv(test_p)
sample_sub = pd.read_csv(sample_sub_p)
N = len(test)
print(f'Test samples: {N:,}')

# Locate inputs
taeyang_p = find_file('s6e9-094649-multi-paradigm-lexsort-master/submission.csv')
talha_p = find_file('s6e9-daily-rank-average-ensemble/submission.csv')
chinzo_p = find_file('s6e9-does-breaking-ties-help/submission.csv')
nathalia_p = find_file('ev-blend/submission.csv')
den_p = find_file('s6e9-nested-te-lightgbm-original-table-dgp-rej/submission_trained.csv')
subodh_p = find_file('s6e9-6-model-meta-stacking-engine/submission.csv')

print('Found all 6 kernel sources successfully!')


In [ ]:
print('2. Loading Predictions & Rank Normalization...')
def load_and_rank(path, name):
    df = pd.read_csv(path)
    target_col = [c for c in df.columns if c != 'id'][0]
    preds = df.set_index('id').loc[sample_sub['id']][target_col].values
    ranks = (rankdata(preds) - 0.5) / N
    print(f'Loaded {name:15s} | Unique: {len(np.unique(preds)):,} | Min: {preds.min():.4f} | Max: {preds.max():.4f}')
    return ranks

rk_taeyang = load_and_rank(taeyang_p, 'Taeyang (0.94650)')
rk_talha   = load_and_rank(talha_p,   'Talha (0.94650)')
rk_chinzo  = load_and_rank(chinzo_p,  'Chinzo (0.94650)')
rk_nath    = load_and_rank(nathalia_p,'Nathalia (0.94650)')
rk_den     = load_and_rank(den_p,     'Den Trained')
rk_subodh  = load_and_rank(subodh_p,  'Subodh Stacker')

# Smart Consensus Weighting
print('\n3. Computing Smart Weighted Ensemble...')
weights = {
    'Taeyang':  0.42,
    'Talha':    0.28,
    'Chinzo':   0.12,
    'Nathalia': 0.10,
    'Den':      0.05,
    'Subodh':   0.03
}

blend = (
    weights['Taeyang']  * rk_taeyang +
    weights['Talha']    * rk_talha +
    weights['Chinzo']   * rk_chinzo +
    weights['Nathalia'] * rk_nath +
    weights['Den']      * rk_den +
    weights['Subodh']   * rk_subodh
)

print(f'Ensemble created with weights: {weights}')


In [ ]:
print('4. Applying Empirical Boundary DGP Corrections...')
inc = pd.to_numeric(test.Annual_Income_USD, errors='coerce').to_numpy()
km = pd.to_numeric(test.Daily_Commute_km, errors='coerce').to_numpy()
env1 = pd.to_numeric(test.Environmental_Concern_Level, errors='coerce').to_numpy() == 1
no_sub = test.Subsidy_Available.astype(str).to_numpy() == 'No'
anx_mh = test.Range_Anxiety_Level.isin(['Medium', 'High']).to_numpy()

shift = np.zeros(N)
# Rule 1: High Income Cliff (100% positive in train data)
shift[inc >= 170537] += 10.0

# Rule 2: Middle-Low Dead Zone (0% positive in train data)
shift[(inc >= 31004) & (inc <= 41970)] -= 10.0

# Rule 3: Extreme Commute Distance (0% positive in train data)
shift[km >= 83] -= 5.0

# Rule 4: 30k Income with No Subsidy + High Anxiety/Low Env (0% positive in train data)
shift[(inc == 30000) & no_sub & (env1 | anx_mh)] -= 5.0

shifted_blend = blend + shift
print(f'Total deterministic rows boundary-shifted: {(shift != 0).sum():,}')

print('\n5. Lexicographical Tie-Breaking...')
def lexrank(primary, secondary):
    order = np.lexsort((secondary, primary))
    ranks = np.empty(len(order), dtype=np.float64)
    ranks[order] = np.arange(1, len(order) + 1)
    return (ranks - 0.5) / len(order)

final_ranks = lexrank(shifted_blend, rk_taeyang)

ties = len(final_ranks) - len(np.unique(final_ranks))
print(f'Ties in final submission: {ties} (Perfect uniform rank ordering)')
print(f'Spearman vs Taeyang Anchor: {spearmanr(final_ranks, rk_taeyang).statistic:.6f}')
print(f'Spearman vs Talha Anchor:   {spearmanr(final_ranks, rk_talha).statistic:.6f}')

submission = pd.DataFrame({
    'id': sample_sub['id'],
    'Will_Buy_EV': final_ranks
})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv successfully!')
print(submission.head(10))


### ðŸš€ Next Steps
This submission combines the highest confidence consensus from all top 0.94650 community solutions while keeping deterministic rule boundary precision intact. Good luck on the leaderboard!
